In [40]:
from typing import Callable

import pandas as pd
import polars as pl
from pathlib import Path
from datetime import datetime, date

def getBatch(inputFunction : Callable) -> Callable:
    def innerWrap(inSeries : pl.Series):
        return pl.Series(map(inputFunction, inSeries))
    return innerWrap

@getBatch
def stripString(x: str | None) -> str | None:
    if x:
        return x.strip()
    return None


@getBatch
def parseDate(x: str) -> date:
    return datetime.strptime(x,"%m-%d-%Y").date()


@getBatch
def parseFloat(x:str) -> float:
    return float(x.replace(",",'')) 

@getBatch
def int2String(x:int | None) -> str | None:
    if x:
        return f"{x}"
    return None

@getBatch
def zeroPadded7(x:str | None) -> str | None:
    if x:
    # temp= 7 - len(x)
        return f"{x:0>7}"
    return None

loc_cols = ['GFPipeID', 'PipeName', 'GFLocID', 'GF_LocID_Tag', 'LocID', 'Loc', 'GF_LocName', 'LocName', 'GF_FacilityType', 'GF_FacilityTypeGroup', 'ReportedFacilityType', 'LocSegment', 'LocZone', 'State', 'County', 'InterconnectingEntity', 'UpdatedTime']

fact_cols =['GasMonth','GasDay','Dataset','GFLocID','LocName','DesignCapacity','OperatingCapacity','TotalScheduledQuantity','OperationallyAvailableCapacity','IT','FlowDirection','Timestamp',]

OA_flow_map =  {
        'Delivery': 'D',
        'Receipt': 'R',
        'Storage Injection': 'D',
        'Storage Withdrawal': 'R'
    }

raw_OA_path = Path('.').absolute().parent / 'downloads/enbridge/OA_raw'

pipe_configs_path = Path('.').absolute().parent / 'src/artifacts/configFiles/PipeConfigs.parquet'

pipesDf = pl.read_parquet(pipe_configs_path)

In [41]:
pipe_configs_path.parent

PosixPath('/Users/saijagadeeshyadavalli/Coding/Particles/GasFundies/GFScrapePackage/src/artifacts/configFiles')

In [42]:
temp_list =[]
for i in raw_OA_path.iterdir():
    pipe_code = i.name.split('_',1)[0].strip()
    temp = pd.read_csv(i)
    if not temp.empty:
        temp_list.append(temp.assign(PipeCode=pipe_code))
    
df_OA : pd.DataFrame= pd.concat(temp_list,ignore_index=True)
df = pl.from_dataframe(df=df_OA.astype(str))

In [43]:
OA_cols_gold = [ 'Eff_Gas_Day', 'Loc', 'Loc_Name', 'Flow_Ind_Desc', 'IT', 'Total_Design_Capacity', 'Operating_Capacity', 'Total_Scheduled_Quantity', 'Operationally_Available_Capacity', 'PipeCode' ] 

OA_gold_map ={
    'Eff_Gas_Day': 'GasDay',
    # 'Cycle_Desc' : 'CycleDesc',
    # 'Loc':'LocID',
    'Loc_Name':'LocName',
    'Flow_Ind_Desc': 'FlowDirection',
    # 'IT':'IT',
    'Total_Design_Capacity':'DesignCapacity',
    'Operating_Capacity':'OperatingCapacity',
    'Total_Scheduled_Quantity':'TotalScheduledQuantity',
    'Operationally_Available_Capacity':'OperationallyAvailableCapacity',
 }

In [44]:
pipesDf.collect_schema()

Schema([('GFPipeID', String),
        ('MetaCode', String),
        ('NoNoticeCode', String),
        ('ParentPipe', String),
        ('PipeCode', String),
        ('PipeName', String),
        ('PointCapCode', String),
        ('SegmentCapCode', String),
        ('StorageCapCode', Null)])

In [45]:
LocsDf = df.select(OA_cols_gold).rename(mapping=OA_gold_map).select(["Loc","LocName","PipeCode"]).join(pipesDf["PipeCode","GFPipeID","PipeName"], on="PipeCode").unique()

LocsDf=LocsDf.with_columns(
    pl.col("Loc").str.strip_chars(),
    pl.col("Loc").str.strip_chars().str.zfill(7).alias("LocID")
    # pl.col("Loc").str.strip_chars().map_batches(zeroPadded7,return_dtype=pl.String).alias("LocID"),
).with_columns(
    pl.concat_str([
        pl.col("GFPipeID"),
        pl.col("LocID")
    ]).alias("GFLocID"),
    pl.lit(datetime.now()).cast(pl.Datetime).alias("UpdatedTime")

)



# LocsDf.write_csv(pipe_configs_path.parent / "OALocMeta.csv")

In [46]:
LocsDf

Loc,LocName,PipeCode,GFPipeID,PipeName,LocID,GFLocID,UpdatedTime
str,str,str,str,str,str,str,datetime[μs]
"""N4GU1""","""(VEC) GUARDIAN""","""NXUS""","""115""","""NEXUS Gas Transmission, LLC (U…","""00N4GU1""","""11500N4GU1""",2026-03-23 09:47:34.636050
"""59079""","""PULASKI""","""ET""","""105""","""East Tennessee Natural Gas, LL…","""0059079""","""1050059079""",2026-03-23 09:47:34.636050
"""70478""","""New Harmony, IL""","""TE""","""120""","""Texas Eastern Transmission, LP""","""0070478""","""1200070478""",2026-03-23 09:47:34.636050
"""839""","""Assonet, Bristol Co, MA""","""AG""","""100""","""Algonquin Gas Transmission, LL…","""0000839""","""1000000839""",2026-03-23 09:47:34.636050
"""59077""","""UCG DUBLIN""","""ET""","""105""","""East Tennessee Natural Gas, LL…","""0059077""","""1050059077""",2026-03-23 09:47:34.636050
…,…,…,…,…,…,…,…
"""N4423""","""(DTE) CONSUMERS NORTHVILLE - …","""NXUS""","""115""","""NEXUS Gas Transmission, LLC (U…","""00N4423""","""11500N4423""",2026-03-23 09:47:34.636050
"""N4HD1""","""(VEC) HARTLAND""","""NXUS""","""115""","""NEXUS Gas Transmission, LLC (U…","""00N4HD1""","""11500N4HD1""",2026-03-23 09:47:34.636050
"""59168""","""PCUD LAFOLLETTE""","""ET""","""105""","""East Tennessee Natural Gas, LL…","""0059168""","""1050059168""",2026-03-23 09:47:34.636050


In [47]:
metaDf =pl.read_csv(pipe_configs_path.parent / "metaLocs.csv")

In [48]:
meta_map = {
    'Loc St Abbrev':'State',
    'Loc Cnty': 'County',
    'Loc Zone':'LocZone',
    'Loc Type Ind':'ReportedFacilityType',
    }

meta_cols_selected = [
    'Loc St Abbrev',
    'Loc Cnty',
    'Loc Zone',
    'Loc Type Ind',
    "GFLocID"
]

In [49]:
loc_cols

['GFPipeID',
 'PipeName',
 'GFLocID',
 'GF_LocID_Tag',
 'LocID',
 'Loc',
 'GF_LocName',
 'LocName',
 'GF_FacilityType',
 'GF_FacilityTypeGroup',
 'ReportedFacilityType',
 'LocSegment',
 'LocZone',
 'State',
 'County',
 'InterconnectingEntity',
 'UpdatedTime']

In [50]:
LocsDf =LocsDf.join(metaDf[meta_cols_selected],on="GFLocID", how="left").rename(mapping=meta_map)

In [51]:
LocsDf.with_columns(
    *[pl.lit(None).cast(pl.String).alias(col_name) for col_name in loc_cols if col_name not in LocsDf.columns]
).select(loc_cols).fill_null("")\
.write_csv(pipe_configs_path.parent / "OALocMeta.csv")

In [52]:
silverOA = df.join(pipesDf["PipeCode","GFPipeID"], on="PipeCode", how="left").with_columns(
    pl.concat_str(
        [
            pl.col("GFPipeID"),
            pl.col("Loc").str.zfill(7)

        ]
    ).alias("GFLocID")
).select(pl.exclude(["PipeCode","GFPipeID"])).unique()

In [53]:
goldOADf=df.select(OA_cols_gold).rename(OA_gold_map).join(pipesDf["GFPipeID","PipeCode"],on="PipeCode",how="left")\
.with_columns(

    pl.col("GasDay").str.to_date(format="%m-%d-%Y"),
    
    pl.col("FlowDirection").str.strip_chars().replace_strict(OA_flow_map, default="N"),
    
    pl.lit(datetime.now()).cast(pl.Datetime).alias("Timestamp"),

    pl.lit("OA").alias("Dataset"),

    pl.col("Loc").str.strip_chars(),

    pl.col("DesignCapacity").str.replace_all(",",""),
    pl.col("OperatingCapacity").str.replace_all(",",""),
    pl.col("TotalScheduledQuantity").str.replace_all(",",""),
    pl.col("OperationallyAvailableCapacity").str.replace_all(",",""),

)\
.with_columns(
    pl.col("Loc").str.zfill(7).alias("LocID"),

    pl.col("DesignCapacity").str.to_decimal(scale=2),
    pl.col("OperatingCapacity").str.to_decimal(scale=2),
    pl.col("TotalScheduledQuantity").str.to_decimal(scale=2),
    pl.col("OperationallyAvailableCapacity").str.to_decimal(scale=2),
)\
.with_columns(
    pl.concat_str([
        pl.col("GFPipeID"),
        pl.col("LocID")
    ]).alias("GFLocID"),
    
    pl.concat_str(
        [
            pl.col("GasDay").dt.year().cast(pl.String),
            pl.col("GasDay").dt.month().cast(pl.String).str.zfill(2)
        ]
    ).str.to_integer().alias("GasMonth")
).select(fact_cols)

goldOADf


GasMonth,GasDay,Dataset,GFLocID,LocName,DesignCapacity,OperatingCapacity,TotalScheduledQuantity,OperationallyAvailableCapacity,IT,FlowDirection,Timestamp
i64,date,str,str,str,"decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]",str,str,datetime[μs]
202603,2026-03-21,"""OA""","""1200070004""","""Dominion Energy Transmission, …",585284.00,590644.00,0.00,590644.00,"""N""","""D""",2026-03-23 09:47:34.702978
202603,2026-03-21,"""OA""","""1200070011""","""Columbia Gas of PA. - Eagle,…",291789.00,444405.00,101136.00,343269.00,"""N""","""D""",2026-03-23 09:47:34.702978
202603,2026-03-21,"""OA""","""1200070018""","""Indiana Gas - Seymour, IN""",16607.00,18422.00,374.00,18048.00,"""N""","""D""",2026-03-23 09:47:34.702978
202603,2026-03-21,"""OA""","""1200070020""","""Equitrans - Waynesburg, PA""",4932.00,5402.00,0.00,5402.00,"""N""","""D""",2026-03-23 09:47:34.702978
202603,2026-03-21,"""OA""","""1200070030""","""Philadelphia Gas Works - Point…",198453.00,198453.00,46451.00,152002.00,"""N""","""D""",2026-03-23 09:47:34.702978
…,…,…,…,…,…,…,…,…,…,…,…
202603,2026-03-22,"""OA""","""1190096107""","""City of Leesburg""",8272.00,8272.00,0.00,8272.00,"""N""","""D""",2026-03-23 09:47:34.702978
202603,2026-03-22,"""OA""","""1190096108""","""TECO Wildwood""",48850.00,48850.00,0.00,48850.00,"""N""","""D""",2026-03-23 09:47:34.702978
202603,2026-03-22,"""OA""","""1190098105""","""TRANSCO ZONE 4 POOL""",1002739.00,1002739.00,555160.00,447579.00,"""N""","""R""",2026-03-23 09:47:34.702978


LocsList = []
for i in pipe_configs_path.parent.iterdir():
    if i.is_file() and i.name.endswith(".csv") and not i.name.startswith("meta"):
        LocsList.append(pd.read_csv(i))

pl.from_dataframe(pd.concat(LocsList,ignore_index=True)).with_columns(
    pl.lit("Enbridge").alias("PartitionKey"),
    pl.col("GFLocID").alias("RowKey"),

).write_csv(pipe_configs_path.parent / "AllLocsInitial.csv")